# Session 13.2: Preparing Image Batches for a CNN Model

**Learning goals:**

- Create image batches using DataLoader.
- Configure appropriate shuffle settings for training and testing.
- Inspect image and label batch shapes and pass one batch to a CNN.

**Prerequisites:**

- Basic CNN training.
- Python with PyTorch.
- Code in Google Colab.

## Outline

0. From tensors to batches
1. DataLoader, batch size and shuffle
2. Live Coding
3. Hands-on Notebook
4. Check for Understanding

## 0. From Tensors to Batches

**Think:** we have 10,000 images to train our CNN.

1. What might happen if we process all 10,000 images at once?
2. What if we process only one image at a time?
3. How could we organise the images to balance memory use and processing efficiency?

All images at once may exceed available memory. One image at a time may not use
parallel processing efficiently. A batch groups several image–label pairs for one model call.

![Four labelled images are shuffled into two batches of two images. Each image keeps its matching label, and one batch at a time is passed to the CNN.](assets/session13-dataloader-batches.png)


## 1. DataLoader, Batch Size and Shuffle

| Setting | Purpose |
|---|---|
| `dataset` | Supply image–label pairs |
| `batch_size=4` | Group up to four samples per batch |
| `shuffle=True` | Randomise training order each epoch |
| `shuffle=False` | Keep test order stable |

Shuffling keeps each image paired with its label. It changes sample order, not
which images belong to the training or test subset.

**Example: create training and test loaders**

Assume `train_dataset` and `test_dataset` are already prepared. Section 2 shows
how to create these subsets and run this example.

```python
from torch.utils.data import DataLoader

BATCH_SIZE = 4
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,  # Randomise training order each epoch.
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,  # Keep test order stable.
)
```

- `train_dataset` and `test_dataset` supply separate image–label pairs.
- `BATCH_SIZE = 4` groups up to four images and their labels in each batch.
- `shuffle=True` gives training a new sample order each epoch; `shuffle=False`
  keeps test order stable for inspection.

Both `train_loader` and `test_loader` provide an **image batch** and its matching
**label batch** together as `(images, labels)`. Each label identifies the class
of the image at the same position in the image batch.

- The image batch, `images`, is the CNN **input tensor**.
- Calling `model(images)` returns the CNN **output tensor**, with one row of
  class logits per image.
- The label batch, `labels`, is the **target tensor** used to compare the CNN
  outputs with the correct classes during training or testing.

| Object | Shape | Meaning |
|---|---|---|
| One image | `[1, 28, 28]` | Channels, height, width |
| Image batch (CNN input tensor) | `[4, 1, 28, 28]` | Batch, channels, height, width |
| Label batch (target tensor) | `[4]` | One correct class label per image |
| CNN output tensor | `[4, 10]` | Ten class logits per image |

[Official documentation: DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

## 2. Live Coding

Prepare the dataset, create two loaders and inspect one batch from each.

### 2.1 Load and split the digit images

In Colab, upload and extract the [digit image bundle](assets/digit_images.zip).

Use the same 80% training and 20% test.


In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split

DIGIT_ROOT = "sample_data/digit_images"
IMAGE_SIZE = (28, 28)

digit_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Keep one intensity channel.
    transforms.Resize(IMAGE_SIZE),  # Set the height and width to 28 pixels.
    transforms.ToTensor(),  # Convert pixels to a float tensor with values in [0, 1].
])
full_dataset = datasets.ImageFolder(DIGIT_ROOT, transform=digit_transform)  # Read labels from class folders.
print("Images:", len(full_dataset), "Classes:", full_dataset.classes)

SEED = 42
torch.manual_seed(SEED) 
TRAIN_FRACTION = 0.8  # Reserve 80% of the images for training.
train_size = int(TRAIN_FRACTION * len(full_dataset))  # Calculate the number of training images.
test_size = len(full_dataset) - train_size  # Use the remaining 20% for testing.
# Randomly assign each image to one subset, keeping training and test images separate.
train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size],  # Return training and test subsets in this order.
)
print("Training images:", len(train_dataset), "Test images:", len(test_dataset))


Images: 240 Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Training images: 192 Test images: 48


### 2.2 Create DataLoaders and inspect a batch

Group four images per batch. Shuffle training images and keep test images in a stable order.

Use `next(iter(loader))` to retrieve one batch. Expect image shape `[4, 1, 28, 28]`
and label shape `[4]`. This inspects one batch, not the whole dataset.


In [2]:
from torch.utils.data import DataLoader

BATCH_SIZE = 4
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,  # Randomise training order each epoch.
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,  # Keep test order stable.
)
print("Training batches:", len(train_loader), "Test batches:", len(test_loader))

images, labels = next(iter(train_loader))  # Retrieve one training batch with matching labels.
test_images, test_labels = next(iter(test_loader))  # Retrieve one test batch.
print("Training image shape:", tuple(images.shape), "Label shape:", tuple(labels.shape))
print("Test image shape:", tuple(test_images.shape), "Label shape:", tuple(test_labels.shape))
print("Training labels:", labels.tolist())


Training batches: 48 Test batches: 12
Training image shape: (4, 1, 28, 28) Label shape: (4,)
Test image shape: (4, 1, 28, 28) Label shape: (4,)
Training labels: [8, 4, 2, 4]


### 2.3 Pass one batch to a CNN

This untrained CNN accepts four images and returns ten class logits per image.

In [3]:
from torch import nn

torch.manual_seed(SEED)  # Reproduce the initial model weights.
NUMBER_OF_CLASSES = len(full_dataset.classes)
POOLED_SIZE = IMAGE_SIZE[0] // 2

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),  # Extract eight feature maps, each 28 × 28.
    nn.ReLU(),  # Apply a non-linear activation.
    nn.MaxPool2d(kernel_size=2),  # Reduce each feature map to 14 × 14.
    nn.Flatten(start_dim=1),  # Flatten features while keeping the batch dimension.
    nn.Linear(8 * POOLED_SIZE * POOLED_SIZE, NUMBER_OF_CLASSES),  # Return ten class logits.
)
model.eval()  # Set evaluation behaviour for the forward pass.
with torch.no_grad():  # Inspect outputs without recording gradients.
    logits = model(images)  # Pass the complete batch to the CNN in one call.
print("CNN output shape:", tuple(logits.shape))

CNN output shape: (4, 10)


## 3. Hands-on Notebook

[Download the practice notebook](<Session13.2 Image Batches Practice.ipynb>){download="Session13.2 Image Batches Practice.ipynb"}

[Download the digit images](assets/digit_images.zip){download="digit_images.zip"}

1. Download both files and open the notebook in [Google Colab](https://colab.research.google.com/) using **File → Upload notebook**.
2. Choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then **Save** and **Connect**.
3. Upload `digit_images.zip` through the **Files** sidebar, then run Sections 1–2 to prepare the data.
4. Write both DataLoader definitions in Section 3, then run Sections 3–5 in order to inspect batches, train the CNN and view test predictions.

With batch size 10, check image shape `[10, 1, 28, 28]` and label shape `[10]`.
After testing, check the combined output shape `[48, 10]` and compare the true and predicted labels.


## 4. Check for Understanding

1. Why use a DataLoader instead of passing all images to the model at once?
2. What do `[4, 1, 28, 28]` and `[4]` represent?
3. Why does this CNN return `[4, 10]` for a batch of four digit images?

Review the pipeline: `Image files → image tensors (transforms) → image batches (DataLoader) → CNN`.

Return to [Session 13](<Session13 Image Classification and Computer Vision Applications.ipynb>) for the cat and dog case study.